In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [16]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.8 MB/s eta 0:00:00


# Audio-to-text transcription for second or additional language learner data

## Importing Libraries

In [19]:
# Data handling
import os
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import csv

# Audio Processing
import librosa
import soundfile as sf
from datasets import Dataset, Audio, load_dataset, DatasetDict

# Core libraries
import torch

# Metric Libraries
import evaluate
import time

# Audio Libraries
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

# setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Loading the Dataset

### Extracting

In [4]:
zip_path = '/content/drive/MyDrive/nonnative.zip'
extract_path = '/content/extracted_data'

# Extraction
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Extracted to: {extract_path}")

Extracted to: /content/extracted_data


### Creating `metadata.csv`

In [8]:
root_dir = "/content/extracted_data"
transcription_file = os.path.join(root_dir, "usma-prompts.txt")
output_csv = os.path.join(root_dir, "metadata.csv")

# Read Transcription file
id_to_text = {}

with open(transcription_file, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        utt_id = parts[0]
        text = parts[1] if len(parts) > 1 else ""
        id_to_text[utt_id] = text

# Create Metadata rows
rows = []

for folder in sorted(os.listdir(root_dir)):
    folder_path = os.path.join(root_dir, folder)

    if not os.path.isdir(folder_path):
        continue

    for file in sorted(os.listdir(folder_path)):
        if not file.endswith(".wav"):
            continue

        utt_id = file.replace(".wav", "")

        if utt_id not in id_to_text:
            print(f"Warning: {utt_id} not found in transcription file")
            continue

        full_path = os.path.join(folder_path, file)

        rows.append({
            "path": full_path,
            "text": id_to_text[utt_id],
            "utterance_id": utt_id
        })

# Write CSV
with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["path", "text", "utterance_id"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Metadata file created at {output_csv}")
print(f"Total samples: {len(rows)}")

Metadata file created at /content/extracted_data/metadata.csv
Total samples: 2028


### Loading with `metadata.csv`

In [9]:
# Loading using metadata.csv
from datasets import load_dataset

metadata_path = "/content/extracted_data/metadata.csv"

dataset = load_dataset(
    "csv",
    data_files=metadata_path,
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
dataset = dataset.rename_column("path", "audio")
print(dataset[0]["audio"]["array"].shape)
print(dataset[0]["audio"]["sampling_rate"])
print(dataset.shape)
print(dataset[:5])

(59827,)
16000
(2028, 3)
{'audio': [<datasets.features._torchcodec.AudioDecoder object at 0x78bcdc890b60>, <datasets.features._torchcodec.AudioDecoder object at 0x78bcdc878c80>, <datasets.features._torchcodec.AudioDecoder object at 0x78bcdc879730>, <datasets.features._torchcodec.AudioDecoder object at 0x78bcdc8797f0>, <datasets.features._torchcodec.AudioDecoder object at 0x78bcdc879880>], 'text': ['vivo en una casa', 'dónde está la estación de tren', 'busqué un regalo para mi mamá', 'javier trabajó allí por seis meses', 'hay una panadería enfrente del museo'], 'utterance_id': ['s1', 's10', 's100', 's101', 's102']}


### Spllitting


| Train | Validation | Test |
|---|---|---|
| 80% | 10% | 10% |



In [22]:
SEED = 42

# First split
train_temp = dataset.train_test_split(
    test_size=0.2,
    seed=SEED
)

# Second Split
val_test = train_temp["test"].train_test_split(
    test_size=0.5,
    seed=SEED
)

dataset_dict = DatasetDict({
    "train": train_temp["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'utterance_id'],
        num_rows: 1622
    })
    validation: Dataset({
        features: ['audio', 'text', 'utterance_id'],
        num_rows: 203
    })
    test: Dataset({
        features: ['audio', 'text', 'utterance_id'],
        num_rows: 203
    })
})


## Fine-Tuning

## Evaluation Metrics

- WER
- RTF